# Semana 9 · Sesión 1: Unidades y álgebra de operadores

**Módulo 2**

## Objetivos de la sesión

1. Hacer cuentas con unidades en `sympy.physics.units`: convertir, usar
   constantes físicas y verificar la consistencia dimensional de una fórmula.
2. Manejar operadores que **no conmutan** con `sympy.physics.quantum`:
   `Operator`, `Dagger` y `Commutator`.
3. Demostrar identidades de conmutadores con `expand(commutator=True)` y
   reconocer operadores hermitianos y unitarios.

## Retomamos

Las últimas semanas SymPy nos dio ecuaciones de movimiento exactas, pero los
símbolos eran números sin dimensión: nada nos impedía sumar metros con
segundos. Hoy atacamos dos cosas que cualquier físico necesita y que ya vienen
en SymPy.

La primera mitad son **unidades**: tratarlas como parte de la expresión, de
modo que las conversiones y los chequeos dimensionales los haga el programa.
La segunda mitad es el **álgebra de operadores** de la mecánica cuántica,
donde $AB \neq BA$ y el orden importa. La próxima sesión la llevamos a espín y
al oscilador armónico.

In [ ]:
import sympy as sp
import sympy.physics.units as u
from sympy.physics.units import convert_to
from sympy.physics.units.systems.si import SI, dimsys_SI
from sympy.physics.quantum import Commutator, Dagger, HermitianOperator, Operator, UnitaryOperator
from sympy.physics.quantum import Bra, Ket, InnerProduct

sp.init_printing()

## Una cantidad es un número con unidad

`sympy.physics.units` define las unidades como símbolos especiales: `u.meter`,
`u.kg`, `u.second`, `u.joule`... Se multiplican por un número y se combinan
con las operaciones de siempre. SymPy no convierte nada por su cuenta: la
energía cinética de una masa de 2 kg a 3 m/s sale en kg·m²/s².

In [ ]:
masa = 2 * u.kg
velocidad = 3 * u.meter / u.second
energia_cinetica = sp.Rational(1, 2) * masa * velocidad**2

display(sp.Eq(sp.Symbol("T"), energia_cinetica))

## `convert_to`: cambiar de unidades

`convert_to(expresion, unidades)` reescribe la expresión en las unidades que
pidas. Si la unidad es compuesta (el joule) lo reconoce; si pides varias,
las usa todas.

In [ ]:
display(sp.Eq(sp.Symbol("T"), convert_to(energia_cinetica, u.joule)))
display(sp.Eq(sp.Symbol("T"), convert_to(energia_cinetica, [u.gram, u.centimeter, u.second])))

Una unidad incompatible no da error: `convert_to` simplemente no puede
convertir y devuelve la expresión sin cambios. Es una forma silenciosa de
detectar que *algo* no cuadra.

In [ ]:
display(convert_to(u.joule, u.meter))

## Constantes físicas

Las constantes del SI viven en el mismo módulo: `u.hbar`, `u.speed_of_light`,
`u.elementary_charge`, `u.electron_rest_mass`, `u.vacuum_permittivity`. Con
`evalf()` salen los números. Un resultado clásico de física de partículas:
$\hbar c \approx 197\ \text{MeV·fm}$, que aquí sale en eV·m.

In [ ]:
hbar_c = u.hbar * u.speed_of_light

display(sp.Eq(sp.Symbol("hbar_c"), convert_to(hbar_c, [u.eV, u.meter]).evalf(6)))

## Dimensiones

Cada unidad tiene una **dimensión**: longitud, masa, tiempo... `dimsys_SI` sabe
descomponer cualquier cantidad en sus dimensiones básicas. Así se ve que la
energía es $M L^2 T^{-2}$.

In [ ]:
display(dimsys_SI.get_dimensional_dependencies(u.energy))

## Verificar una fórmula

Un chequeo dimensional es la prueba más barata de una fórmula. Tomemos el
periodo del péndulo, $T = 2\pi\sqrt{\ell/g}$. Sustituimos cada símbolo por
una cantidad con su unidad y le pedimos a SymPy la dimensión del resultado con
`SI._collect_factor_and_dimension` (el guion bajo indica que es una función
interna, pero es la que usa el propio tutorial de SymPy). Devuelve el factor
numérico y la dimensión.

In [ ]:
largo, g = sp.symbols("ell g", positive=True)  # longitud y aceleración de la gravedad: positivas
periodo = 2 * sp.pi * sp.sqrt(largo / g)

valores = {largo: 1 * u.meter, g: 9.8 * u.meter / u.second**2}
factor, dimension = SI._collect_factor_and_dimension(periodo.subs(valores))

display(sp.Eq(sp.Symbol("T"), periodo))
display(dimension)

Si la fórmula estuviera mal escrita, `_collect_factor_and_dimension` lo
dice con una excepción. Probemos con una "fórmula" que suma una longitud a un
tiempo. Es un error a propósito, así que va dentro de `try/except`.

In [ ]:
try:
    SI._collect_factor_and_dimension(1 * u.meter + 1 * u.second)
except ValueError as error:
    print("Error dimensional:", error)

## TODO en clase 1

El **radio de Bohr** se puede construir con las constantes del SI:

$$a_0 = \frac{4\pi\,\varepsilon_0\,\hbar^2}{m_e\, e^2}.$$

1. Escribe `radio_bohr` con `u.vacuum_permittivity`, `u.hbar`,
   `u.electron_rest_mass` y `u.elementary_charge`.
2. Verifica que su dimensión es **longitud**, con
   `SI._collect_factor_and_dimension`.
3. Conviértelo a metros y a ångström (`u.angstrom`), con `evalf()`. Debe salir
   aproximadamente $0.529\ \text{Å}$.

*Pista:* `u.hbar` incluye internamente a $\pi$; el resultado numérico lo
absorbe.

In [ ]:
# TODO en clase: el radio de Bohr
radio_bohr = ...
dimension_radio = ...
radio_en_metros = ...
radio_en_angstrom = ...

## Operadores que no conmutan

En `sympy.physics.quantum` un `Operator("A")` es un símbolo **no
conmutativo**: SymPy no reordena `A*B` a `B*A`. Eso es justo lo que pide la
mecánica cuántica. `Dagger` es el adjunto (transpuesta conjugada), y recuerda
la regla $(AB)^\dagger = B^\dagger A^\dagger$.

In [ ]:
A = Operator("A")
B = Operator("B")

display(A * B - B * A)
display(Dagger(A * B))      # SymPy ya invirtió el orden
display(Dagger(sp.I * A))

Observa que el $i$ sale **conjugado**: $(iA)^\dagger = -iA^\dagger$.

## `Commutator`

`Commutator(A, B)` representa $[A, B] = AB - BA$ sin evaluarlo. `doit()` lo
expande a su definición, y `expand(commutator=True)` aplica las reglas de
linealidad y de Leibniz para sacar los operadores de adentro.

In [ ]:
conmutador = Commutator(A, B)

display(sp.Eq(conmutador, conmutador.doit(), evaluate=False))
display(Commutator(A, A).doit())

In [ ]:
C_ = Operator("C")  # el guion bajo evita pisar la C de Python

producto = Commutator(A * B, C_)
display(sp.Eq(producto, producto.expand(commutator=True), evaluate=False))

La identidad que SymPy acaba de mostrar es la regla de Leibniz para
conmutadores: $[AB, C] = A[B, C] + [A, C]B$. Es la herramienta de trabajo para
todos los cálculos con $[x, p] = i\hbar$.

## Kets, bras y productos internos

Un `Ket("psi")` es el vector de estado $|\psi\rangle$; su `Dagger` es el bra.
El producto de un bra con un ket es un `InnerProduct`, y los operadores
`HermitianOperator` y `UnitaryOperator` ya traen sus propiedades:
$H^\dagger = H$ y $U^\dagger U = 1$.

In [ ]:
psi = Ket("psi")
phi = Ket("phi")

display(Dagger(psi))
display(Bra("phi") * psi)
display(Dagger(Bra("phi") * psi))

In [ ]:
H = HermitianOperator("H")
U = UnitaryOperator("U")

display(Dagger(H))          # un hermitiano es su propio adjunto
display(Dagger(U) * U)

## TODO en clase 2

Dos demostraciones con el álgebra de conmutadores.

1. **Identidad de Jacobi.** Con `A`, `B` y `C_` como arriba, define `jacobi`
   como la suma cíclica $[A,[B,C]] + [B,[C,A]] + [C,[A,B]]$ y compruébala:
   `expand(commutator=True)` seguido de `doit()` y `expand()` debe dar `0`.
2. **El $i$ que lo hace hermitiano.** Con `P` y `Q` hermitianos
   (`HermitianOperator`), el conmutador $[P,Q]$ es *antihermitiano*, y por eso
   $i[P,Q]$ sí es hermitiano. Calcula `Dagger(sp.I * Commutator(P, Q).doit())`
   y restale `sp.I * Commutator(P, Q).doit()`: tras `expand()` debe dar `0`.

In [ ]:
# TODO en clase: Jacobi y el conmutador hermitiano
jacobi = ...
jacobi_expandido = ...

P = ...
Q = ...
diferencia_hermitiana = ...

## Resumen

Con `sympy.physics.units` una cantidad lleva su unidad: `convert_to` cambia de
unidades, las constantes del SI están a la mano, y `dimsys_SI` entrega las
dimensiones, de modo que podemos **verificar una fórmula** antes de creerle.
`convert_to` falla en silencio (deja la expresión intacta) y
`_collect_factor_and_dimension` falla con ruido.

Con `sympy.physics.quantum` los operadores son símbolos no conmutativos.
`Dagger` invierte el orden y conjuga; `Commutator` se expande con
`expand(commutator=True)` y se evalúa con `doit()`; y `HermitianOperator` y
`UnitaryOperator` traen sus propiedades.

**Siguiente sesión:** operadores *concretos*. Espín y momento angular con
`quantum.spin`, matrices de Pauli, producto tensorial y el oscilador armónico
con operadores de escalera, donde `qapply` aplica operadores a estados.